# 03 — Feature Engineering

**Primary author:** Victoria

**Builds on:**
- *02_embedding_generation.ipynb* (Victoria — embedding files and index contract)
- Hans's *Hans_Supervised_Learning.ipynb* and *Hans_Supervised_Learning_Models.ipynb*
  (cosine similarity computation pattern, WordNet relationship features, surface
  features — expanding from 10 to 46)

**Prompt engineering:** Victoria  
**AI assistance:** Claude Code (Anthropic)  
**Environment:** Local (CPU only)

Compute all 46 features for each (clue, definition, answer) row. Loads
embeddings from Step 2, computes 15 context-free + 6 context-informed cosine
similarities, 21 WordNet relationship features, and 4 surface features.
Outputs `data/features_all.parquet`.

Implements PLAN.md Step 3.

**Inputs:**
- `data/clues_filtered.csv` (Step 1)
- `data/embeddings/` — 3 `.npy` arrays + 3 index CSVs (Step 2)
- `data/embeddings/clue_context_phrases.csv` (Step 2 — provides `definition_wn`
  and `answer_wn` lookup keys)
- WordNet (via NLTK)

**Output:**
- `data/features_all.parquet` — one row per (clue, definition, answer) with
  all 46 features plus metadata columns

## Imports

In [ ]:
import warnings

import numpy as np
import pandas as pd
from pathlib import Path

import nltk
from nltk.corpus import wordnet as wn
from sklearn.metrics.pairwise import cosine_similarity

from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)

## Environment and Paths

In [ ]:
# --- Environment Auto-Detection ---
# Same pattern as 02_embedding_generation.ipynb: detect Colab, Great Lakes,
# or local and set paths accordingly.
try:
    IS_COLAB = 'google.colab' in str(get_ipython())
except NameError:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/SIADS 692 Milestone II/'
                        'Milestone II - NLP Cryptic Crossword Clues/'
                        'clue_misdirection')
else:
    # Local or Great Lakes: notebook is in clue_misdirection/notebooks/,
    # so parent is the clue_misdirection project root.
    try:
        PROJECT_ROOT = Path(__file__).resolve().parent.parent
    except NameError:
        PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / 'data'
EMBEDDINGS_DIR = DATA_DIR / 'embeddings'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Download WordNet data — needed for the 21 relationship features computed
# later in this notebook.
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print(f'Environment: {"Google Colab" if IS_COLAB else "Local / Great Lakes"}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Data directory: {DATA_DIR}')
print(f'Embeddings directory: {EMBEDDINGS_DIR}')

## Load Data

We load two CSV files:

1. **`clues_filtered.csv`** (Step 1) — 241,397 rows with all metadata columns
   (`clue_id`, `clue`, `surface`, `definition`, `answer`, `def_answer_pair_id`,
   etc.).

2. **`clue_context_phrases.csv`** (Step 2) — 240,211 rows that survived the
   Step 2 cleanup (dropping 1,186 rows for duplicate definitions in the surface
   text and fully unresolvable words). This file provides the `definition_wn`
   and `answer_wn` columns — the WordNet-ready lookup keys needed to find the
   correct row in the embedding index files.

We inner-merge on `(clue_id, definition)` to produce a 240,211-row working
set that has both the original metadata and the embedding lookup keys.
Merging on `clue_id` alone would be wrong: double-definition clues have
multiple rows per `clue_id` (one per valid definition), so a `clue_id`-only
join would produce a many-to-many cross product. Adding `definition` as a
second key disambiguates these rows.

We also record each row's position in `clue_context_phrases.csv` as
`cc_row_position`. Since `clue_context_phrases.csv` and
`clue_context_embeddings.npy` are in identical row order (verified in
Step 2), this gives us a direct index into the clue-context embedding
array — no need to map through `clue_context_index.csv`, which only has
`clue_id` and would suffer the same double-definition ambiguity.

In [ ]:
# --- Load clues_filtered.csv (Step 1 output) ---
clues_path = DATA_DIR / 'clues_filtered.csv'
assert clues_path.exists(), (
    f'Missing input file: {clues_path}\n'
    f'Run 01_data_cleaning.ipynb first to produce this file.'
)
clues_df = pd.read_csv(clues_path)
print(f'clues_filtered.csv: {len(clues_df):,} rows')

# --- Load clue_context_phrases.csv (Step 2 intermediate) ---
# This file provides definition_wn and answer_wn — the WordNet-ready lookup
# keys that map to the embedding index files. It also identifies which rows
# survived Step 2's cleanup (240,211 of the original 241,397).
# CRITICAL: keep_default_na=False prevents pandas from interpreting the word
# "nan" (grandmother) as NaN — see DATA.md.
cc_phrases_path = EMBEDDINGS_DIR / 'clue_context_phrases.csv'
assert cc_phrases_path.exists(), (
    f'Missing input file: {cc_phrases_path}\n'
    f'Run 02_embedding_generation.ipynb first to produce this file.'
)
cc_phrases = pd.read_csv(cc_phrases_path, keep_default_na=False)
print(f'clue_context_phrases.csv: {len(cc_phrases):,} rows')

# --- Record each row's position in cc_phrases / clue_context_embeddings ---
# clue_context_phrases.csv and clue_context_embeddings.npy are in identical
# row order (verified in Step 2). By recording the row position here, we get
# a direct index into the clue-context embedding array after the merge —
# avoiding the ambiguity of mapping through clue_context_index.csv, which
# only has clue_id and can't disambiguate double-definition clues.
cc_phrases['cc_row_position'] = np.arange(len(cc_phrases))

# --- Merge to get definition_wn and answer_wn onto the clue rows ---
# Inner merge restricts to the 240,211 rows that have embeddings.
# We merge on (clue_id, definition) — NOT clue_id alone — because
# double-definition clues have multiple rows per clue_id. A clue_id-only
# merge would produce a many-to-many cross product for those clues,
# inflating the row count. The 'definition' column appears in both files
# (original-case definition text) and disambiguates which definition each
# row corresponds to.
df = clues_df.merge(
    cc_phrases[['clue_id', 'definition', 'definition_wn', 'answer_wn',
                'def_num_usable_synsets', 'ans_num_usable_synsets',
                'cc_row_position']],
    on=['clue_id', 'definition'],
    how='inner'
)

# Verify the merge produced exactly the expected number of rows — no
# inflation from many-to-many joins and no unexpected drops.
assert len(df) == len(cc_phrases), (
    f'Merge produced {len(df):,} rows, expected {len(cc_phrases):,}. '
    f'This likely means a double-definition clue was not disambiguated '
    f'correctly by the (clue_id, definition) key.')

print(f'\nWorking set after merge: {len(df):,} rows')
print(f'  (dropped {len(clues_df) - len(df):,} rows without embeddings)')
print(f'  Unique (definition, answer) pairs: '
      f'{df["def_answer_pair_id"].nunique():,}')
print(f'\nColumns: {list(df.columns)}')
df.head(3)

## Load Embeddings

Load all 6 embedding files from Step 2 (3 `.npy` arrays + 3 index CSVs).
The index CSVs map row positions in the `.npy` arrays to word strings
(`definition_wn` / `answer_wn`) or `clue_id`.

In [ ]:
# Load embedding arrays and index files.
# CRITICAL: keep_default_na=False on all index CSVs — the word "nan"
# (grandmother) is a valid crossword definition/answer.

definition_embeddings = np.load(EMBEDDINGS_DIR / 'definition_embeddings.npy')
definition_index = pd.read_csv(
    EMBEDDINGS_DIR / 'definition_index.csv', index_col=0,
    keep_default_na=False)

answer_embeddings = np.load(EMBEDDINGS_DIR / 'answer_embeddings.npy')
answer_index = pd.read_csv(
    EMBEDDINGS_DIR / 'answer_index.csv', index_col=0,
    keep_default_na=False)

clue_context_embeddings = np.load(
    EMBEDDINGS_DIR / 'clue_context_embeddings.npy')
clue_context_index = pd.read_csv(
    EMBEDDINGS_DIR / 'clue_context_index.csv', index_col=0,
    keep_default_na=False)

# --- Print shapes and sizes ---
EMBED_DIM = 1024
print(f'{"File":<35s} {"Shape":<25s} {"Memory":>8s}')
print(f'{"-"*35} {"-"*25} {"-"*8}')
for name, arr in [
    ('definition_embeddings.npy', definition_embeddings),
    ('answer_embeddings.npy', answer_embeddings),
    ('clue_context_embeddings.npy', clue_context_embeddings),
]:
    mb = arr.nbytes / 1024**2
    print(f'{name:<35s} {str(arr.shape):<25s} {mb:>6.1f} MB')

total_mb = (definition_embeddings.nbytes + answer_embeddings.nbytes
            + clue_context_embeddings.nbytes) / 1024**2
print(f'\nTotal embedding memory: {total_mb:.1f} MB')
print(f'\nIndex sizes:')
print(f'  definition_index:     {len(definition_index):,} rows')
print(f'  answer_index:         {len(answer_index):,} rows')
print(f'  clue_context_index:   {len(clue_context_index):,} rows')

# --- Shape and consistency assertions ---
n_def = len(definition_index)
n_ans = len(answer_index)
n_cc = len(clue_context_index)

assert definition_embeddings.shape == (n_def, 3, EMBED_DIM), (
    f'definition_embeddings shape mismatch: expected ({n_def}, 3, {EMBED_DIM}), '
    f'got {definition_embeddings.shape}')
assert answer_embeddings.shape == (n_ans, 3, EMBED_DIM), (
    f'answer_embeddings shape mismatch: expected ({n_ans}, 3, {EMBED_DIM}), '
    f'got {answer_embeddings.shape}')
assert clue_context_embeddings.shape == (n_cc, EMBED_DIM), (
    f'clue_context_embeddings shape mismatch: expected ({n_cc}, {EMBED_DIM}), '
    f'got {clue_context_embeddings.shape}')

print(f'\nAll shape assertions passed.')

## Build Embedding Matrices

For each of the 240,211 rows in our working set, we need 7 embedding vectors
(each 1024-dim):

| Abbrev | Source | Slot in `.npy` array |
|--------|--------|---------------------|
| `w1_all` | definition allsense-average | `definition_embeddings[idx, 0, :]` |
| `w1_common` | definition common synset | `definition_embeddings[idx, 1, :]` |
| `w1_obscure` | definition obscure synset | `definition_embeddings[idx, 2, :]` |
| `w2_all` | answer allsense-average | `answer_embeddings[idx, 0, :]` |
| `w2_common` | answer common synset | `answer_embeddings[idx, 1, :]` |
| `w2_obscure` | answer obscure synset | `answer_embeddings[idx, 2, :]` |
| `w1_clue` | definition in clue context | `clue_context_embeddings[idx, :]` |

Rather than looping row-by-row, we build full `(N, 1024)` matrices for each
embedding type using bulk index lookups, then fancy-index into the `.npy`
arrays. For definition and answer embeddings, we map `definition_wn` /
`answer_wn` strings to their row positions in the index files. For
clue-context embeddings, we use the `cc_row_position` column recorded during
the merge — this is a direct positional index into the `.npy` array and
correctly handles double-definition clues (where multiple rows share the
same `clue_id` but have different clue-context embeddings).

In [ ]:
# --- Build word → row-position mappings for O(1) lookup ---
# definition_index and answer_index have integer row indices (0, 1, 2, ...)
# and a 'word' column. We create a Series mapping word string → row position.
def_word_to_idx = pd.Series(
    definition_index.index, index=definition_index['word'])
ans_word_to_idx = pd.Series(
    answer_index.index, index=answer_index['word'])

# --- Map each row's lookup key to its position in the embedding arrays ---
# .map() returns NaN for any key not found — we assert none are missing.
def_indices = df['definition_wn'].map(def_word_to_idx)
assert def_indices.notna().all(), (
    f'{def_indices.isna().sum()} definition_wn values not found in '
    f'definition_index. Examples: '
    f'{df.loc[def_indices.isna(), "definition_wn"].head().tolist()}')
def_indices = def_indices.astype(int).values

ans_indices = df['answer_wn'].map(ans_word_to_idx)
assert ans_indices.notna().all(), (
    f'{ans_indices.isna().sum()} answer_wn values not found in '
    f'answer_index. Examples: '
    f'{df.loc[ans_indices.isna(), "answer_wn"].head().tolist()}')
ans_indices = ans_indices.astype(int).values

# For clue-context embeddings, we use cc_row_position directly. This column
# was set to np.arange(len(cc_phrases)) before the merge, giving each row
# its position in clue_context_embeddings.npy. Unlike clue_context_index.csv
# (which only has clue_id), cc_row_position correctly handles double-definition
# clues where multiple rows share the same clue_id but have different
# clue-context embeddings.
cc_indices = df['cc_row_position'].values

# --- Fancy-index into embedding arrays to build (N, 1024) matrices ---
# Each matrix has one row per clue in our working set.
N = len(df)
print(f'Building 7 embedding matrices of shape ({N:,}, {EMBED_DIM})...')

w1_all     = definition_embeddings[def_indices, 0, :]   # (N, 1024)
w1_common  = definition_embeddings[def_indices, 1, :]   # (N, 1024)
w1_obscure = definition_embeddings[def_indices, 2, :]   # (N, 1024)
w2_all     = answer_embeddings[ans_indices, 0, :]       # (N, 1024)
w2_common  = answer_embeddings[ans_indices, 1, :]       # (N, 1024)
w2_obscure = answer_embeddings[ans_indices, 2, :]       # (N, 1024)
w1_clue    = clue_context_embeddings[cc_indices, :]     # (N, 1024)

print(f'Done. Each matrix: {w1_all.shape}')
print(f'Total memory for 7 matrices: '
      f'{7 * w1_all.nbytes / 1024**2:.1f} MB')

## Cosine Similarity Features (21 total)

We compute 21 pairwise cosine similarities organized into two groups:

**Context-Free Meaning (15):** All ${6 \choose 2} = 15$ pairwise cosine
similarities among the 6 context-free embeddings (3 definition × 3 answer
cross-word pairs = 9, plus 3 within-definition and 3 within-answer pairs).
These capture the semantic relationship between definition and answer
senses *without* any influence from the clue's surface reading.

**Context-Informed Meaning (6):** Cosine similarity between
`word1_clue_context` (the definition embedded within the clue surface
using CALE's `<t></t>` delimiters) and each of the 6 context-free
embeddings. These capture how the clue's surface reading shifts the
definition's embedding — the core signal for misdirection.

We use row-wise dot products on L2-normalized vectors rather than
`sklearn.metrics.pairwise.cosine_similarity` on individual pairs. This
computes all N similarities for a given pair of embedding types in one
vectorized operation, which is orders of magnitude faster than looping.

In [ ]:
def rowwise_cosine(A, B):
    """Compute row-wise cosine similarity between corresponding rows of A and B.

    Parameters
    ----------
    A, B : np.ndarray, shape (N, D)
        Two matrices with the same shape.

    Returns
    -------
    np.ndarray, shape (N,)
        Cosine similarity for each row pair: cos(A[i], B[i]).
    """
    # L2-normalize each row, then take the row-wise dot product.
    A_norm = np.linalg.norm(A, axis=1, keepdims=True)
    B_norm = np.linalg.norm(B, axis=1, keepdims=True)
    return np.sum((A / A_norm) * (B / B_norm), axis=1)


# --- Context-Free Meaning: 15 features ---
# Cross-word pairs (definition × answer): 3 × 3 = 9
df['cos_w1all_w2all']       = rowwise_cosine(w1_all, w2_all)
df['cos_w1all_w2common']    = rowwise_cosine(w1_all, w2_common)
df['cos_w1all_w2obscure']   = rowwise_cosine(w1_all, w2_obscure)
df['cos_w1common_w2all']    = rowwise_cosine(w1_common, w2_all)
df['cos_w1common_w2common'] = rowwise_cosine(w1_common, w2_common)
df['cos_w1common_w2obscure']= rowwise_cosine(w1_common, w2_obscure)
df['cos_w1obscure_w2all']   = rowwise_cosine(w1_obscure, w2_all)
df['cos_w1obscure_w2common']= rowwise_cosine(w1_obscure, w2_common)
df['cos_w1obscure_w2obscure'] = rowwise_cosine(w1_obscure, w2_obscure)

# Within-definition pairs: C(3,2) = 3
df['cos_w1all_w1common']    = rowwise_cosine(w1_all, w1_common)
df['cos_w1all_w1obscure']   = rowwise_cosine(w1_all, w1_obscure)
df['cos_w1common_w1obscure']= rowwise_cosine(w1_common, w1_obscure)

# Within-answer pairs: C(3,2) = 3
df['cos_w2all_w2common']    = rowwise_cosine(w2_all, w2_common)
df['cos_w2all_w2obscure']   = rowwise_cosine(w2_all, w2_obscure)
df['cos_w2common_w2obscure']= rowwise_cosine(w2_common, w2_obscure)

# --- Context-Informed Meaning: 6 features ---
# word1_clue_context paired with each of the 6 context-free embeddings.
df['cos_w1clue_w1all']      = rowwise_cosine(w1_clue, w1_all)
df['cos_w1clue_w1common']   = rowwise_cosine(w1_clue, w1_common)
df['cos_w1clue_w1obscure']  = rowwise_cosine(w1_clue, w1_obscure)
df['cos_w1clue_w2all']      = rowwise_cosine(w1_clue, w2_all)
df['cos_w1clue_w2common']   = rowwise_cosine(w1_clue, w2_common)
df['cos_w1clue_w2obscure']  = rowwise_cosine(w1_clue, w2_obscure)

print('Computed 21 cosine similarity features.')

## Verification: Cosine Similarity Features

In [ ]:
# --- Identify the 21 cosine columns ---
cos_cols = [c for c in df.columns if c.startswith('cos_')]

# Assert exactly 21 cosine similarity columns
assert len(cos_cols) == 21, (
    f'Expected 21 cosine similarity columns, found {len(cos_cols)}: {cos_cols}')

# Assert no NaN values in any cosine column (Decision 3: no NaN features).
# Cosine similarities are guaranteed non-NaN as long as embeddings are non-zero,
# which we verified in Step 2.
nan_counts = df[cos_cols].isnull().sum()
assert (nan_counts == 0).all(), (
    f'NaN values found in cosine columns:\n'
    f'{nan_counts[nan_counts > 0]}')

print(f'Verification passed: 21 cosine columns, 0 NaN values.')
print(f'Rows: {len(df):,}\n')

# --- Descriptive statistics ---
# Context-free features (15): cross-word similarities tend to be moderate
# (definition and answer are semantically related but not identical), while
# within-word similarities tend to be high (different senses of the same word
# still share much of the embedding space).
# Context-informed features (6): the clue context shifts the definition
# embedding — the degree of shift is the misdirection signal.
print('--- Descriptive Statistics (21 Cosine Similarity Features) ---\n')
stats = df[cos_cols].describe().T[['mean', 'std', 'min', 'max']]
# Use wider display so columns don't wrap
with pd.option_context('display.float_format', '{:.4f}'.format,
                       'display.max_colwidth', 30):
    print(stats.to_string())

print(f'\n--- First 5 Rows (Cosine Similarity Features) ---\n')
with pd.option_context('display.float_format', '{:.4f}'.format,
                       'display.max_columns', None):
    display(df[cos_cols].head())